In [15]:
import pandas as pd
import numpy as np
import xgboost as xgb
import joblib
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

In [16]:
import pandas as pd


print("1. Ingesting 2024 Gold Layer Data...")

# Load the raw TLC Parquet file (Super fast and memory efficient!)
raw_trips = pd.read_parquet("data/yellow_tripdata_2024-01.parquet")

# Load the dimension tables
dim_location = pd.read_csv("data/dim_localizacao.csv")
dim_time = pd.read_csv("data/dim_data.csv")
dim_weather = pd.read_csv("data/dim_clima.csv")

1. Ingesting 2024 Gold Layer Data...


In [17]:
print("2. Translating Dimension Schemas to English...")

dim_location.rename(columns={
    'localizacao_id': 'LocationID',
    'bairro': 'Borough',
    'zona': 'Zone',
    'zona_servico': 'service_zone'
}, inplace=True)

dim_time.rename(columns={
    'data_id': 'date_id',
    'data': 'date',
    'ano': 'year',
    'mes': 'month',
    'dia': 'day',
    'dia_semana': 'day_of_week'
}, inplace=True)

dim_weather.rename(columns={
    'clima_id': 'weather_id',
    'data': 'date',
    'precipitacao': 'precip',
    'temp_max': 'temp_max',
    'temp_min': 'temp_min',
    'neve': 'snow'
}, inplace=True)

2. Translating Dimension Schemas to English...


In [18]:
print("3. Flattening with Native TLC Datetimes...")

df = raw_trips.copy()

# 1. Ensure Datetime Objects
df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])
df['tpep_dropoff_datetime'] = pd.to_datetime(df['tpep_dropoff_datetime'])

# 2. Extract the High-Value Features!
df['trip_time'] = (df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']).dt.total_seconds()
df['request_hour'] = df['tpep_pickup_datetime'].dt.hour
df['date'] = df['tpep_pickup_datetime'].dt.strftime('%Y-%m-%d')

# Rename the TLC distance column to match your pipeline
df.rename(columns={'trip_distance': 'trip_miles'}, inplace=True)

# 3. Merge Locations
df = df.merge(dim_location, left_on="PULocationID", right_on="LocationID", how="inner")
df.rename(columns={'Borough': 'PU_Borough'}, inplace=True)
df.drop(columns=['LocationID'], inplace=True)

df = df.merge(dim_location, left_on="DOLocationID", right_on="LocationID", how="inner")
df.rename(columns={'Borough': 'DO_Borough'}, inplace=True)
df.drop(columns=['LocationID'], inplace=True)

# 4. Merge Weather & Calendar mapping
df = df.merge(dim_time, on="date", how="left")
df = df.merge(dim_weather, on="date", how="left")

print("✅ Join Complete.")

3. Flattening with Native TLC Datetimes...
✅ Join Complete.


In [19]:
print("4. Filtering, Downcasting, and Engineering...")

# 1. CREATE TEMP FIRST (Fixes the KeyError!)
df['temp'] = ((df['temp_max'] + df['temp_min']) / 2.0).astype('float32')

# 2. FILTER FIRST (Throw away bad rows)
df = df[(df['trip_miles'] > 0.1) & (df['trip_miles'] < 100)]
df = df[(df['fare_amount'] > 2.50) & (df['fare_amount'] < 400)]
df = df[(df['trip_time'] > 60)] # Drop trips under 1 minute (ghost trips)
df.dropna(subset=['fare_amount', 'trip_miles', 'temp'], inplace=True)

# 3. FAST ROUTE ID & FLAGS
df['route_id'] = (df['PULocationID'] * 1000) + df['DOLocationID']
airport_zones = [132, 138, 1]
df['is_airport_trip'] = (df['PULocationID'].isin(airport_zones) | df['DOLocationID'].isin(airport_zones)).astype('int8')
df['is_weekend'] = df['day_of_week'].isin([1, 7]).astype('int8')

# 4. API Pydantic Alignment
df['shared_match_flag'] = np.int8(0)
df['wav_match_flag'] = np.int8(0)
df['access_a_ride_flag'] = np.int8(0)

# 5. MEMORY DOWNCASTING
float_cols = df.select_dtypes(include=['float64']).columns
int_cols = df.select_dtypes(include=['int64']).columns
df[float_cols] = df[float_cols].astype('float32')
df[int_cols] = df[int_cols].astype('int32')
df['request_hour'] = df['request_hour'].astype('int8')

# 6. MEMORY-SAFE ONE HOT ENCODING
df = pd.get_dummies(df, columns=['PU_Borough', 'DO_Borough'], drop_first=False, dtype='uint8')

print(f"✅ Fast Engineering Complete. Matrix shape: {df.shape}")

4. Filtering, Downcasting, and Engineering...
✅ Fast Engineering Complete. Matrix shape: (224456, 55)


In [20]:
# 1. Calculate Implied Speed (Miles per Hour)
# Formula: (miles / seconds) * 3600 seconds in an hour
df['implied_speed_mph'] = (df['trip_miles'] / df['trip_time']) * 3600

# 2. Clean dirty data (e.g., GPS glitches that say a taxi went 300mph or 0mph for 10 miles)
df['implied_speed_mph'] = df['implied_speed_mph'].clip(lower=0, upper=75)

# 3. Handle any potential division-by-zero NaN values
df['implied_speed_mph'] = df['implied_speed_mph'].fillna(0)

# Now proceed with training your XGBoost model as usual!
# Make sure to include 'implied_speed_mph' in your feature list.

In [ ]:
print("5. Chronological Splitting & Leakage-Free Target Encoding...")

# Sort by actual pickup time to strictly prevent data leakage
df.sort_values('tpep_pickup_datetime', inplace=True)

# Define final features matching the API payload
features = [
    'PULocationID', 'DOLocationID', 'trip_miles', 'trip_time',
    'shared_match_flag', 'wav_match_flag', 'access_a_ride_flag',
    'request_hour', 'is_weekend', 'is_airport_trip', 
    'temp', 'precip', 'snow','implied_speed_mph'
]

# Dynamically grab the One-Hot Encoded Borough columns
borough_cols = [c for c in df.columns if 'PU_Borough_' in c or 'DO_Borough_' in c]
features.extend(borough_cols)

# Chronological 80/20 Split
split_idx = int(len(df) * 0.8)
train_df = df.iloc[:split_idx].copy()
test_df = df.iloc[split_idx:].copy()

# Calculate both the mean and the count of trips for each route
route_stats = train_df.groupby('route_id')['fare_amount'].agg(['mean', 'count'])
global_mean = train_df['fare_amount'].mean()

# Bayesian Smoothing Factor (m) - the number of rides required to fully trust the route's mean
m = 50 

# Apply the smoothing formula
route_stats['smoothed_mean'] = (
    (route_stats['count'] * route_stats['mean']) + (m * global_mean)
) / (route_stats['count'] + m)

route_means = route_stats['smoothed_mean'].astype('float32').to_dict()

# Map the smoothed values
train_df['historical_route_fare'] = train_df['route_id'].map(route_means).astype('float32')
test_df['historical_route_fare'] = test_df['route_id'].map(route_means).fillna(global_mean).astype('float32')

features.append('historical_route_fare')

# FINALIZE MATRICES
X_train = train_df[features]
y_train = train_df['fare_amount']
X_test = test_df[features]
y_test = test_df['fare_amount']

print(f"✅ Split Complete! Training Rows: {len(X_train):,}, Testing Rows: {len(X_test):,}")

5. Chronological Splitting & Leakage-Free Target Encoding...
✅ Split Complete! Training Rows: 179,564, Testing Rows: 44,892


In [22]:
X_train.columns

Index(['PULocationID', 'DOLocationID', 'trip_miles', 'trip_time',
       'shared_match_flag', 'wav_match_flag', 'access_a_ride_flag',
       'request_hour', 'is_weekend', 'is_airport_trip', 'temp', 'precip',
       'snow', 'implied_speed_mph', 'PU_Borough_Bronx', 'PU_Borough_Brooklyn',
       'PU_Borough_EWR', 'PU_Borough_Manhattan', 'PU_Borough_Queens',
       'PU_Borough_Staten Island', 'DO_Borough_Bronx', 'DO_Borough_Brooklyn',
       'DO_Borough_EWR', 'DO_Borough_Manhattan', 'DO_Borough_Queens',
       'DO_Borough_Staten Island', 'historical_route_fare'],
      dtype='str')

In [23]:
print("6. Training Optimized 2024 XGBoost Model...")
model = xgb.XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method='hist',
    n_jobs=-1,
    random_state=42
)

model.fit(X_train, y_train)
print("✅ Training complete.")

6. Training Optimized 2024 XGBoost Model...
✅ Training complete.


In [24]:
print("7. Evaluating Accuracy...")
predictions = model.predict(X_test)
mae = mean_absolute_error(y_test, predictions)

print(f"Mean Absolute Error (MAE): ${mae:.2f}")
print(f"Baseline (Global Mean) MAE: ${mean_absolute_error(y_test, [global_mean_fare]*len(y_test)):.2f}")

joblib.dump(model, "production_taxi_master_model_2024.joblib")
print("✅ SUCCESS: 2024 Model saved as production_taxi_model_2024.joblib")

7. Evaluating Accuracy...
Mean Absolute Error (MAE): $0.85
Baseline (Global Mean) MAE: $13.08
✅ SUCCESS: 2024 Model saved as production_taxi_model_2024.joblib


In [25]:
import json

# Convert float32 to standard float for JSON serialization
export_means = {str(k): float(v) for k, v in route_means.items()}

with open('route_means.json', 'w') as f:
    json.dump(export_means, f)
    
print("✅ Exported route_means.json to API!")

✅ Exported route_means.json to API!
